In [7]:
%load_ext autoreload
%autoreload 2
%cd /mnt/sdd1/akanksha/formulacode/datasmith
import datetime

import pandas as pd

from datasmith.logging_config import get_logger

logger = get_logger("notebooks.building_reports_pr")

curr_date: str = datetime.datetime.now().isoformat()

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
/mnt/sdd1/akanksha/formulacode/datasmith


/mnt/sdd1/akanksha/formulacode/datasmith/.venv/lib/python3.10/site-packages/IPython/core/magics/osm.py:417: UserWarning: This is now an optional IPython functionality, setting dhist requires you to install the `pickleshare` library.
  self.shell.db['dhist'] = compress_dhist(dhist)[-100:]


In [8]:
commits_df = pd.read_parquet(
    "/mnt/sdd1/atharvas/formulacode/datasmith/scratch/artifacts/pipeflush/less_filtered_commits_perfonly.parquet"
)

In [9]:
from datasmith.execution.collect_commits import collect_merge_shas

commits = collect_merge_shas(repo="astropy/astropy")

Paginating GitHub:  38%|███▊      | 38/100 [01:09<01:52,  1.82s/page]
20:24:27 INFO     datasmith: Collected 3141 merged PR SHAs (non-null) from astropy/astropy.


In [4]:
from tqdm.auto import tqdm

from datasmith.scrape.build_pr_report import build_pr_report

reports = []


# with ThreadPoolExecutor(max_workers=100) as executor:
#     futures = {
#         executor.submit(
#             build_pr_report,
#             link=commit['url'],
#             summarize_llm=False,
#             add_classification=False,
#         ): commit for commit in commits[:750]
#     }
#     for future in tqdm(as_completed(futures), total=len(futures)):
#         commit = futures[future]
#         try:
#             report = future.result()
#             reports.append(report)
#         except Exception as e:
#             logger.error(f"Error processing commit {commit['url']}: {e}")
# above as a for loop:

for commit in tqdm(commits[:100]):
    if not commit:
        reports.append(None)
        continue
    report = build_pr_report(
        link=commit["url"],
        summarize_llm=False,
        add_classification=False,
    )
    reports.append(report)

0it [00:00, ?it/s]


In [5]:
df = pd.DataFrame([{**c, **{"report": report}} for c, report in zip(commits, reports)])

In [6]:
df[["url", "report"]][~df["report"].str.contains("NOT_A_VALID_PR")].url.values

KeyError: "None of [Index(['url', 'report'], dtype='object')] are in the [columns]"

In [ ]:
# logger.info(f"PR Report:\n{report}")
print(report)

NOT_A_VALID_PR
